In [ ]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2 — Clone your repo
!git clone https://github.com/bodasingiksheeraja317-svg/streamsense.git
%cd streamsense

fatal: destination path 'streamsense' already exists and is not an empty directory.
/content/streamsense/training/streamsense


In [ ]:
# Diagnostic — run this before anything else
import os, zipfile

# Check 1: did the zip extract at all?
print("=== Contents of /content/streamsense/data/ ===")
data_path = '/content/streamsense/data'
if os.path.exists(data_path):
    for item in os.listdir(data_path):
        print(f"  {item}")
else:
    print("  /content/streamsense/data/ does not exist")

# Check 2: what is inside the zip?
zip_path = '/content/drive/MyDrive/data_raw.zip'
print(f"\n=== First 20 entries inside {zip_path} ===")
with zipfile.ZipFile(zip_path, 'r') as z:
    for name in list(z.namelist())[:20]:
        print(f"  {name}")

=== Contents of /content/streamsense/data/ ===
  raw
  splits

=== First 20 entries inside /content/drive/MyDrive/data_raw.zip ===
  raw\down\00176480_nohash_0.wav
  raw\down\004ae714_nohash_0.wav
  raw\down\00b01445_nohash_0.wav
  raw\down\00b01445_nohash_1.wav
  raw\down\00f0204f_nohash_0.wav
  raw\down\012187a4_nohash_0.wav
  raw\down\012187a4_nohash_1.wav
  raw\down\0132a06d_nohash_0.wav
  raw\down\0132a06d_nohash_1.wav
  raw\down\0132a06d_nohash_2.wav
  raw\down\0132a06d_nohash_3.wav
  raw\down\0132a06d_nohash_4.wav
  raw\down\0137b3f4_nohash_0.wav
  raw\down\0137b3f4_nohash_1.wav
  raw\down\0137b3f4_nohash_2.wav
  raw\down\0137b3f4_nohash_3.wav
  raw\down\0137b3f4_nohash_4.wav
  raw\down\014f9f65_nohash_0.wav
  raw\down\0165e0e8_nohash_0.wav
  raw\down\016e2c6d_nohash_0.wav


In [ ]:
# Fix — extract directly to the correct location
import zipfile, os

zip_path    = '/content/drive/MyDrive/data_raw.zip'
extract_to  = '/content/streamsense/data/'

# Make sure target exists
os.makedirs(extract_to, exist_ok=True)

print("Extracting... this will take 2-3 minutes")
with zipfile.ZipFile(zip_path, 'r') as z:
    # Fix backslashes from Windows zip → forward slashes for Linux
    for member in z.infolist():
        # Convert Windows path separators
        member.filename = member.filename.replace('\\', '/')
        z.extract(member, extract_to)

print("\nDone. Checking:")
for cls in sorted(os.listdir('/content/streamsense/data/raw')):
    count = len(os.listdir(f'/content/streamsense/data/raw/{cls}'))
    print(f"  {cls}: {count} files")

Extracting... this will take 2-3 minutes

Done. Checking:
  down: 3917 files
  go: 3880 files
  left: 3801 files
  no: 3941 files
  off: 3745 files
  on: 3845 files
  right: 3778 files
  stop: 3872 files
  up: 3723 files
  yes: 4044 files


In [ ]:
# Cell 5 — Verify split files exist
from pathlib import Path

splits = [
    '/content/streamsense/data/splits/train_files.txt',
    '/content/streamsense/data/splits/val_files.txt',
    '/content/streamsense/data/splits/test_files.txt',
]
for s in splits:
    p = Path(s)
    lines = p.read_text().strip().splitlines()
    print(f"{p.name}: {len(lines)} lines — {'OK' if lines else 'EMPTY'}")

train_files.txt: 26984 lines — OK
val_files.txt: 5783 lines — OK
test_files.txt: 5779 lines — OK


In [ ]:
import re

splits = [
    '/content/streamsense/data/splits/train_files.txt',
    '/content/streamsense/data/splits/val_files.txt',
    '/content/streamsense/data/splits/test_files.txt',
]

for split_path in splits:
    with open(split_path, 'r') as f:
        content = f.read()

    # Remove the Windows drive + STREAMSENSE prefix regardless of slash
    # style/count (C:, C:/, C:\, C://, C:\\ ... followed by STREAMSENSE
    # and any number of slashes/backslashes)
    content = re.sub(r'[Cc]:[\\/]+STREAMSENSE[\\/]+', '/content/streamsense/', content)

    # Collapse any remaining backslashes or double-slashes to single forward slash
    content = content.replace('\\', '/')
    content = re.sub(r'/+', '/', content)

    with open(split_path, 'w') as f:
        f.write(content)
    print(f"Rewritten: {split_path}")

print()
!head -3 /content/streamsense/data/splits/train_files.txt

Rewritten: /content/streamsense/data/splits/train_files.txt
Rewritten: /content/streamsense/data/splits/val_files.txt
Rewritten: /content/streamsense/data/splits/test_files.txt

/content/streamsense/data/raw/yes/bbd0bbd0_nohash_4.wav | yes | 0
/content/streamsense/data/raw/yes/aa80f517_nohash_4.wav | yes | 0
/content/streamsense/data/raw/yes/e71a9381_nohash_4.wav | yes | 0


In [ ]:
!ls /content/streamsense/training/

dataset_1d.py		   mel_pipeline.py  streamsense  verify_pipeline.py
dataset.py		   model_1d.py	    train_1d.py
evaluate_1d_comparison.py  model.py	    train.py


In [ ]:
# ── Cell 6: Set STREAMSENSE_ROOT to match your actual extraction path ────────
import os
os.environ["STREAMSENSE_ROOT"] = "/content/streamsense"

# Sanity check — this must print the same path your zip extracted to
print("STREAMSENSE_ROOT =", os.environ["STREAMSENSE_ROOT"])


STREAMSENSE_ROOT = /content/streamsense


In [ ]:
# ── Cell 7: Verify GPU is available ────────────────────────────────────────────
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

CUDA available: True
Device: Tesla T4


In [ ]:
# ── Cell 8: Make sure training dependencies are present ───────────────────────
# Colab ships with torch/torchaudio preinstalled, but versions can drift.
# This confirms torchaudio (needed for WAV loading) works correctly.

import torchaudio
print("torchaudio:", torchaudio.__version__)

# Quick load test on one real file from your dataset
import os
from pathlib import Path
raw_dir = Path(os.environ["STREAMSENSE_ROOT"]) / "data" / "raw"
first_class = sorted(os.listdir(raw_dir))[0]
first_file = sorted(os.listdir(raw_dir / first_class))[0]
test_path = raw_dir / first_class / first_file

waveform, sr = torchaudio.load(str(test_path))
print(f"Loaded {test_path.name}: shape={waveform.shape}, sample_rate={sr}")


torchaudio: 2.11.0+cu128
Loaded 00176480_nohash_0.wav: shape=torch.Size([1, 16000]), sample_rate=16000


In [ ]:
# ── Cell 9: cd into training/ and run smoke tests ─────────────────────────────

%cd /content/streamsense/training

!python model_1d.py

!python dataset_1d.py

/content/streamsense/training
StreamSenseNet1D — smoke test
Total parameters: 591,210
Input  shape: (4, 1, 16000)
Output shape: (4, 10)

StreamSenseNet (2D)  : 295,786 params
StreamSenseNet1D (1D): 591,210 params

[PASS] Smoke test OK.
StreamSenseDataset1D — smoke test
  train:  26984 samples  sample0 shape=(1, 16000) dtype=torch.float32 label=0
  val  :   5783 samples  sample0 shape=(1, 16000) dtype=torch.float32 label=0
  test :   5779 samples  sample0 shape=(1, 16000) dtype=torch.float32 label=0

[PASS] Smoke test OK.


In [ ]:
!python train_1d.py

STREAMSENSE — train_1d.py (Epic A3.2, 1D CNN baseline)
Device: cuda
Seed:   42

Loading datasets...
  train: 26984 samples
  val  : 5783 samples

Model: StreamSenseNet1D
  Parameters: 591,210

Training (max 60 epochs, early stop patience=8)...
 Epoch  TrainLoss  TrainAcc   ValLoss   ValAcc         LR   Time
     1     1.6398    41.41%    1.4487   44.94%   1.00e-03  65.7s
         -> new best (val_acc=44.94%), saved to best_model_1d.pth
     2     1.0393    65.31%    0.7065   77.57%   1.00e-03  66.4s
         -> new best (val_acc=77.57%), saved to best_model_1d.pth
     3     0.7464    76.25%    0.5108   83.88%   1.00e-03  65.4s
         -> new best (val_acc=83.88%), saved to best_model_1d.pth
     4     0.6068    80.53%    0.4342   86.06%   1.00e-03  65.8s
         -> new best (val_acc=86.06%), saved to best_model_1d.pth
     5     0.5218    83.41%    0.3956   86.77%   1.00e-03  65.8s
         -> new best (val_acc=86.77%), saved to best_model_1d.pth
     6     0.4799    84.74%    0.405

In [ ]:
%cd /content/streamsense/training
!python evaluate_1d_comparison.py

/content/streamsense/training
STREAMSENSE — evaluate_1d_comparison.py (Epic A3.2)
Loaded checkpoint: epoch 53, val_acc=95.97%
Parameters: 591,210
Test samples: 5779

Running inference...
Inference time: 4.43s for 5779 samples (0.77 ms/sample)
Test accuracy: 95.76%
Test loss:     0.1396

Saved -> /content/streamsense/evaluation_1d/evaluation_report_1d.txt
Saved -> /content/streamsense/evaluation_1d/confusion_matrix_1d.png
Saved -> /content/streamsense/evaluation_1d/comparison_1d_vs_2d.txt

SUMMARY
  2D StreamSenseNet : 295,786 params, 95.97% test acc
  1D StreamSenseNet1D: 591,210 params, 95.76% test acc

[DONE] See comparison_1d_vs_2d.txt for full comparison.


In [ ]:
import os
print(os.path.exists('/content/streamsense/checkpoints/best_model.pth'))

True


In [ ]:
with open('/content/streamsense/evaluation_1d/comparison_1d_vs_2d.txt') as f:
    print(f.read())

STREAMSENSE — Architecture Comparison: 1D CNN (raw) vs 2D CNN (mel)
Supports Epic A3.3 (ADR — Architecture Decision Record)

Metric                         2D StreamSenseNet   1D StreamSenseNet1D
----------------------------------------------------------------------
Parameters                               295,786               591,210
Test accuracy                             95.97%                95.76%
Test loss                                 0.1273                0.1396
Input representation           log-mel [1,64,97]raw waveform [1,16000]

Per-class accuracy (%):
Class             2D        1D   Delta (1D-2D)
----------------------------------------------
yes           98.84%    98.02%          -0.82%
no            96.79%    93.91%          -2.88%
up            95.17%    95.71%          +0.54%
down          94.04%    95.91%          +1.87%
left          96.67%    94.56%          -2.11%
right         99.29%    98.59%          -0.70%
on            95.66%    96.35%          +0.69%
o

In [ ]:
%cd /content/streamsense

!git config user.email "bodasingiksheeraja317@gmail.com"
!git config user.name "bodasingiksheeraja317-svg"

!git add checkpoints_1d/ evaluation_1d/
!git commit -m "Add A3.2 1D CNN baseline trained checkpoint + evaluation (Colab GPU, val_acc=95.97%)"

!git push https://bodasingiksheeraja317-svg:<YOUR_NEW_TOKEN>@github.com/bodasingiksheeraja317-svg/streamsense.git main

/content/streamsense
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   data/splits/test_files.txt
	modified:   data/splits/train_files.txt
	modified:   data/splits/val_files.txt

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	training/streamsense/

no changes added to commit (use "git add" and/or "git commit -a")
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 2.14 MiB | 5.07 MiB/s, done.
Total 9 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/bodasingiksheeraja317-svg/streamsense.git
   bdcac55..c4